# Semana 25 · ¿Cuánto tarda la luz en llegar?

Este cuaderno acompaña a la sesión de proyecto de la semana 25: una calculadora de tiempo de luz y una simulación del descubrimiento de Rømer.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Una calculadora de tiempo de luz

Guardamos las constantes en variables con nombre y escribimos una función que acepta una distancia en km, ua o años luz y devuelve el tiempo que tarda la luz, expresado de forma legible.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

C = 299_792.458            # km/s, exacta por definición
UA = 1.495978707e8         # km
DIA = 86_400               # s
AÑO = 365.25 * DIA         # s (año juliano)
AÑO_LUZ = C * AÑO          # km

def tiempo_de_luz(distancia, unidad="km"):
    """Segundos que tarda la luz en recorrer la distancia dada."""
    factores = {"km": 1.0, "ua": UA, "al": AÑO_LUZ}
    return distancia * factores[unidad] / C

def legible(s):
    """Convierte segundos en un texto con la unidad más cómoda."""
    if s < 60:        return f"{s:.2f} s"
    if s < 3600:      return f"{s/60:.1f} min"
    if s < DIA:       return f"{s/3600:.2f} h"
    if s < AÑO:       return f"{s/DIA:.1f} días"
    if s < 1e6 * AÑO: return f"{s/AÑO:,.1f} años"
    return f"{s/AÑO/1e6:,.2f} millones de años"

print("1 año luz =", f"{AÑO_LUZ:.4e} km =", f"{AÑO_LUZ/UA:,.0f} ua")
for nombre, d, u in [("Luna", 384_400, "km"), ("Sol", 1, "ua"), ("Júpiter (oposición)", 4.2, "ua"),
                     ("Voyager 1 (nov. 2026)", 173.1, "ua"), ("Próxima Centauri", 4.24, "al"),
                     ("Sirio", 8.6, "al"), ("Andrómeda", 2.5e6, "al")]:
    print(f"{nombre:22s} → {legible(tiempo_de_luz(d, u))}")

## 2. Todas las distancias en una gráfica

Los tiempos van de un segundo a millones de años: más de 13 órdenes de magnitud. En una escala normal, todo menos Andrómeda quedaría aplastado en el cero. En escala logarítmica (semana 15) cada marca vale diez veces la anterior.

In [ ]:
objetos = {"Luna": (384_400, "km"), "Sol": (1, "ua"), "Júpiter": (4.2, "ua"), "Neptuno": (30, "ua"),
           "Voyager 1": (173.1, "ua"), "Próxima Cen.": (4.24, "al"), "Vega": (25, "al"),
           "Pléyades": (440, "al"), "Centro galáctico": (26_000, "al"), "Andrómeda": (2.5e6, "al")}
nombres = list(objetos)
tiempos = [tiempo_de_luz(*objetos[n]) for n in nombres]

plt.figure(figsize=(9, 4.5))
plt.barh(nombres, tiempos)
plt.xscale("log")
for ref, etiqueta in [(1, "1 s"), (60, "1 min"), (3600, "1 h"), (DIA, "1 día"), (AÑO, "1 año"), (1e6 * AÑO, "1 Ma")]:
    plt.axvline(ref, color="gray", lw=0.5, ls="--")
    plt.text(ref, len(nombres) - 0.4, etiqueta, fontsize=8, ha="center")
plt.xlabel("tiempo que tarda la luz en llegar (s, escala logarítmica)")
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

**Ejercicio.** Añade a la gráfica la estrella Polar (unos 430 años luz) y la galaxia M81, en la Osa Mayor (unos 12 millones de años luz).

In [ ]:
# Tu código aquí

In [ ]:
# Solución
for nombre, d in [("Polar", 430), ("M81", 12e6)]:
    print(f"{nombre}: {legible(tiempo_de_luz(d, 'al'))}")

## 3. Los eclipses de Ío, como los vio Rømer

Simulamos la Tierra y Júpiter en órbitas circulares alrededor del Sol (como en la semana 13) y los eclipses de Ío, que ocurren con un ritmo perfectamente regular: uno cada 1,7699 días (el periodo de Ío respecto a la dirección del Sol, que es la que marca la sombra de Júpiter).

Cada eclipse ocurre en el instante $t_k$, pero lo vemos en $t_k + d/c$, donde $d$ es la distancia Tierra-Júpiter en ese momento. Simulamos dos años.

In [ ]:
T_TIERRA, T_JUPITER = 365.256, 4332.59     # días
R_TIERRA, R_JUPITER = 1.0, 5.203           # ua
P_ECLIPSES = 1.7699                        # días

def posicion(r, T, t, fase=0.0):
    ang = fase + 2 * np.pi * t / T
    return np.array([r * np.cos(ang), r * np.sin(ang)])

# Empezamos con Júpiter en oposición: los dos planetas en la misma dirección
t_real = np.arange(0, 730, P_ECLIPSES)          # instantes reales de los eclipses (días)
dist_ua = np.array([np.linalg.norm(posicion(R_JUPITER, T_JUPITER, t) - posicion(R_TIERRA, T_TIERRA, t)) for t in t_real])
retraso_s = dist_ua * UA / C                    # segundos que tarda la luz
t_observado = t_real + retraso_s / DIA          # días

print(f"Distancia Tierra-Júpiter: entre {dist_ua.min():.2f} y {dist_ua.max():.2f} ua")
print(f"Tiempo de luz: entre {retraso_s.min()/60:.1f} y {retraso_s.max()/60:.1f} minutos")

Rømer no conocía estos instantes reales. Lo que podía hacer era medir el periodo de los eclipses cerca de la oposición y predecir con él los siguientes. Hacemos lo mismo: usamos los diez primeros eclipses observados para calcular el periodo aparente, predecimos todos los demás con ese periodo y miramos la diferencia entre lo observado y lo predicho (los astrónomos la llaman **O − C**, *observado menos calculado*).

In [ ]:
P_medido = (t_observado[10] - t_observado[0]) / 10
prediccion = t_observado[0] + P_medido * np.arange(len(t_observado))
o_menos_c = (t_observado - prediccion) * DIA / 60     # en minutos

plt.figure(figsize=(10, 4))
plt.plot(t_real, o_menos_c, ".", ms=3)
plt.xlabel("días desde la oposición"); plt.ylabel("observado − predicho (min)")
plt.title("Los eclipses de Ío se retrasan y se adelantan con la distancia a Júpiter")
plt.grid(alpha=0.3); plt.show()
print(f"Periodo aparente medido cerca de la oposición: {P_medido:.5f} días (real: {P_ECLIPSES})")

Los eclipses se van retrasando respecto a la predicción hasta que Júpiter está al otro lado del Sol, y se vuelven a adelantar al acercarse la siguiente oposición. El retraso máximo sale algo mayor que los 16,6 minutos que tarda la luz en cruzar la órbita terrestre, porque el periodo medido cerca de la oposición ya está un poco afectado por el alejamiento de la Tierra, y ese pequeño error se acumula eclipse tras eclipse.

## 4. Recuperar la velocidad de la luz

Si el retraso de cada eclipse es $d/c$, al dibujar el retraso frente a la distancia sale una recta de pendiente $1/c$. Un ajuste lineal (semana 15) nos da la pendiente.

In [ ]:
# El retraso de cada eclipse respecto a su instante real, frente a la distancia en km
retraso = (t_observado - t_real) * DIA
d_km = dist_ua * UA
pendiente, ordenada = np.polyfit(d_km, retraso, 1)
print(f"Pendiente: {pendiente:.4e} s/km   →   c = {1/pendiente:,.0f} km/s")

plt.plot(d_km / 1e6, retraso / 60, ".", ms=3)
plt.xlabel("distancia Tierra-Júpiter (millones de km)"); plt.ylabel("retraso (min)")
plt.grid(alpha=0.3); plt.show()

Aquí hemos hecho trampa: conocíamos los instantes reales. Rømer solo tenía el O − C. **Ejercicio.** Usa solo la amplitud del O − C (la diferencia entre su máximo y su mínimo) y el diámetro de la órbita terrestre (2 ua) para estimar $c$, como hizo Huygens.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
amplitud_s = (o_menos_c.max() - o_menos_c.min()) * 60
c_estimada = 2 * UA / amplitud_s
print(f"Amplitud del O − C: {amplitud_s/60:.1f} min   →   c ≈ {c_estimada:,.0f} km/s")

Sale unos 212 000 km/s, curiosamente cerca de la estimación de Huygens. El valor se queda corto porque la amplitud del O − C mezcla el efecto de la distancia con el error acumulado del periodo medido: el mismo tipo de problema que tenían Rømer y Huygens, además de la imprecisión de los relojes de péndulo de 1676.

## 5. Experimenta

1. Haz que Júpiter estuviera a 10 ua del Sol. ¿Cambia la amplitud del retraso? ¿Por qué no?
2. Con Saturno (9,54 ua, periodo 10 759 días) y su luna Titán (eclipses cada 15,95 días aproximadamente), ¿se vería el mismo efecto? ¿Con qué amplitud?
3. Calcula cuántos días tardaría una nave a la velocidad de la Voyager 1 (unos 17 km/s) en recorrer un segundo luz.

In [ ]:
# Tu experimento aquí